# 3.3 — Nâng cấp DL trên folds chung

Kết quả 3.1/3.2: MLP baseline OOF 0.94256 thua cả Spline LR (0.94469) → nút thắt là cách MLP biểu diễn cột số.
Các thí nghiệm (mỗi cái ghi `oof/<name>.csv`, `test/<name>.csv`):

| Tên | Thay đổi so với `mlp_base` |
|---|---|
| `mlp_fe` | + 9 feature của ML (n_flags, 6 decimal fractions, 2 interactions); tối đa 20 epoch |
| `mlp_plr` | PLR embedding cho 22 cột số (QuantileTransformer → sin/cos → Linear → ReLU), embedding cho 5 category + 5 cột rời rạc, MLP 256→128, OneCycle lr 2e-3, batch 1024, 12 epoch, `sigma=0.1` |
| `mlp_plr_sig1` | như `mlp_plr`, `sigma=1.0` (tần số cao hơn) |

Vòng 1 (đã chạy): `mlp_plr_sig1` tốt nhất — OOF 0.94516, blend gain +0.00011; best epoch 4–7/12 → overfit sau đỉnh LR.
Vòng 2 (`RUN` mặc định):

| Tên | Thay đổi so với `mlp_plr_sig1` |
|---|---|
| `mlp_plr_sig3` | `sigma=3.0` |
| `mlp_plr_sig1_reg` | dropout 0.30/0.20, weight decay 1e-2, lr 1e-3, 10 epoch |
| `mlp_plr_sig3_reg` | `sigma=3.0` + regularize như trên |

Vòng 2 (đã chạy): `mlp_plr_sig3_reg` tốt nhất cho blend — OOF 0.94544, Spearman với lgb_nl3 0.955, gain +0.00049 (alpha ~0.34).
σ lớn hơn → khác ML nhiều hơn. Vòng 3 (`RUN` mặc định):

| Tên | Thay đổi |
|---|---|
| `mlp_plr_sig10_reg` | `sigma=10.0` + regularize (1 seed) — xem xu hướng σ còn tăng không |
| `mlp_plr_sig3_reg_x3` | `mlp_plr_sig3_reg` trung bình 3 seed (9, 10, 11) |

Ô cuối cùng gom dự đoán của các phiên bản trước vào output, để phiên bản mới nhất của 3.3 chứa đủ mọi model DL cho 3.4.

Vòng 3 (đã chạy): `mlp_plr_sig3_reg_x3` OOF 0.94707, gain +0.00057; σ=10 quá cao. Ensemble v1 (3.4): LB 0.94972.

**Vòng 4** — chọn bằng biến `ROUND`:

| ROUND | Huấn luyện | Ghép thành (trung bình có trọng số theo số seed) |
|---|---|---|
| `4a` | `mlp_plr_sig3_reg` seed 12, 13; `mlp_plr_sig3` seed 10, 11; `mlp_plr_sig10_reg` seed 10, 11 | `mlp_plr_sig3_reg_x5`, `mlp_plr_sig3_x3`, `mlp_plr_sig10_reg_x3` |
| `4b` | `mlp_plr_sig2_reg`, `mlp_plr_sig5_reg`, `resnet_plr_sig3` (ResNet head, σ=3, regularize) | — |

Ghép seed: x5 = (3 · x3 + 2 · bản 2 seed mới) / 5 — giống hệt chạy lại 5 seed vì OOF/test của mỗi bản đã là trung bình xác suất theo seed.

Ô cuối so với ML: AUC, tương quan Spearman với `lgb_nl3`, và lợi ích khi blend với ML (trọng số DL chọn bằng CV theo fold).

**Input:** competition + `ccf-shared-code` + output 3.0 + output 3.1 + output 3.2 + output phiên bản trước của chính notebook 3.3. **Accelerator:** GPU T4.

In [ ]:
import sys
from pathlib import Path

# shared/ code: local repo or the Kaggle dataset that holds features.py + dl.py
cands = [Path("shared/features.py"), Path("../../shared/features.py"), *Path("/kaggle/input").rglob("features.py")]
feat = next(p for p in cands if p.is_file())
sys.path.insert(0, str(feat.parent.resolve()))
import features as F
print("shared code:", feat.parent)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

tr, te, y, train_id, test_id = F.load()
fold = F.load_folds(train_id)          # from notebook 3.0 output (folds_5_seed9.csv)
assert len(tr) == F.N_TRAIN and len(te) == F.N_TEST and set(np.unique(fold)) == set(range(F.N_FOLDS))
log(f"train {tr.shape}, test {te.shape}, fraud rate {y.mean():.4%}")

In [ ]:
import torch
import dl as D
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, StandardScaler

print("Device:", D.DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# Thí nghiệm sẽ chạy. Sau khi chọn được cấu hình tốt nhất, chạy lại với SEEDS = [9, 10, 11]
ROUND = "4a"          # "4a" rồi "4b" (mỗi lần một phiên bản notebook)

# (thí nghiệm, seeds, tên output)
RUNS = {
    "4a": [("mlp_plr_sig3_reg", [12, 13], "mlp_plr_sig3_reg_s12_13"),
           ("mlp_plr_sig3", [10, 11], "mlp_plr_sig3_s10_11"),
           ("mlp_plr_sig10_reg", [10, 11], "mlp_plr_sig10_reg_s10_11")],
    "4b": [("mlp_plr_sig2_reg", [9], "mlp_plr_sig2_reg"),
           ("mlp_plr_sig5_reg", [9], "mlp_plr_sig5_reg"),
           ("resnet_plr_sig3", [9], "resnet_plr_sig3")],
}
# tên output mới -> [(tên có sẵn, số seed), ...]
MERGES = {
    "4a": {"mlp_plr_sig3_reg_x5": [("mlp_plr_sig3_reg_x3", 3), ("mlp_plr_sig3_reg_s12_13", 2)],
           "mlp_plr_sig3_x3": [("mlp_plr_sig3", 1), ("mlp_plr_sig3_s10_11", 2)],
           "mlp_plr_sig10_reg_x3": [("mlp_plr_sig10_reg", 1), ("mlp_plr_sig10_reg_s10_11", 2)]},
    "4b": {},
}
RUN, MERGE = RUNS[ROUND], MERGES[ROUND]
# Kết quả các lần chạy trước (gắn output của 3.2 và phiên bản trước của 3.3), để so sánh ở ô cuối
PREVIOUS = ["mlp_base", "mlp_plr_sig3", "mlp_plr_sig1_reg", "mlp_plr_sig3_reg", "mlp_plr_sig10_reg",
            "mlp_plr_sig3_reg_x3", "mlp_plr_sig3_reg_x5", "mlp_plr_sig3_x3", "mlp_plr_sig10_reg_x3"]

EX_tr, EX_te = F.add_extra_features(tr), F.add_extra_features(te)
TR = pd.concat([tr, EX_tr], axis=1)
TE = pd.concat([te, EX_te], axis=1)
EXTRA = list(EX_tr.columns)
PLR_NUM = F.CONTINUOUS + F.DISCRETE + EXTRA       # 22 cột số -> PLR
EMB_CAT = F.CATS + F.DISCRETE                     # 10 cột -> nn.Embedding
print("extra:", EXTRA)

In [ ]:
# ---------- feature builders: fit on train fold only, return (train, valid, test) tuples of arrays + model factory
def build_onehot(a, b):
    pre = ColumnTransformer([
        ("num", StandardScaler(), F.NUMS + F.BOOLS + EXTRA),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), F.CATS),
    ]).fit(TR.iloc[a])
    arr = [pre.transform(d).astype(np.float32) for d in (TR.iloc[a], TR.iloc[b], TE)]
    return [(x,) for x in arr], lambda: D.MLP(arr[0].shape[1])


def build_plr(a, b, sigma, dropout=(0.15, 0.10), head="mlp"):
    qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200_000,
                             random_state=F.SEED).fit(TR.iloc[a][PLR_NUM])
    num = [qt.transform(d[PLR_NUM]).astype(np.float32) for d in (TR.iloc[a], TR.iloc[b], TE)]
    binary = [d[F.BOOLS].to_numpy(np.float32) for d in (TR.iloc[a], TR.iloc[b], TE)]
    ca, cb, ct, card = D.encode_categories(TR.iloc[a][EMB_CAT].astype(str), TR.iloc[b][EMB_CAT].astype(str),
                                           TE[EMB_CAT].astype(str))
    arrays = [(n, bi, c) for n, bi, c in zip(num, binary, (ca, cb, ct))]
    return arrays, lambda: D.EmbeddingMLP(len(PLR_NUM), len(F.BOOLS), card, sigma=sigma, dropout=dropout, head=head)


PLR_CFG = D.TrainConfig(lr=2e-3, weight_decay=1e-4, batch_size=1024, max_epochs=12, patience=12,
                        scheduler="onecycle", grad_clip=1.0)
REG_CFG = D.TrainConfig(lr=1e-3, weight_decay=1e-2, batch_size=1024, max_epochs=10, patience=10,
                        scheduler="onecycle", grad_clip=1.0)
REG_DROPOUT = (0.30, 0.20)
EXPERIMENTS = {
    # vòng 1
    "mlp_fe": (build_onehot, D.TrainConfig(max_epochs=20, patience=3)),
    "mlp_plr": (lambda a, b: build_plr(a, b, sigma=0.1), PLR_CFG),
    "mlp_plr_sig1": (lambda a, b: build_plr(a, b, sigma=1.0), PLR_CFG),
    # vòng 2
    "mlp_plr_sig3": (lambda a, b: build_plr(a, b, sigma=3.0), PLR_CFG),
    "mlp_plr_sig1_reg": (lambda a, b: build_plr(a, b, sigma=1.0, dropout=REG_DROPOUT), REG_CFG),
    "mlp_plr_sig3_reg": (lambda a, b: build_plr(a, b, sigma=3.0, dropout=REG_DROPOUT), REG_CFG),
    # vòng 3
    "mlp_plr_sig10_reg": (lambda a, b: build_plr(a, b, sigma=10.0, dropout=REG_DROPOUT), REG_CFG),
    # vòng 4b
    "mlp_plr_sig2_reg": (lambda a, b: build_plr(a, b, sigma=2.0, dropout=REG_DROPOUT), REG_CFG),
    "mlp_plr_sig5_reg": (lambda a, b: build_plr(a, b, sigma=5.0, dropout=REG_DROPOUT), REG_CFG),
    "resnet_plr_sig3": (lambda a, b: build_plr(a, b, sigma=3.0, dropout=(0.20,), head="resnet"), REG_CFG),
}

In [ ]:
results = {}
for name, SEEDS, out_name in RUN:
    build, cfg = EXPERIMENTS[name]
    oof = np.zeros(len(tr))
    test_pred = np.zeros(len(te))
    fold_info = []
    for f, a, b in F.iter_folds(fold):
        (Xa, Xb, Xt), make_model = build(a, b)
        for s in SEEDS:
            t0 = time.time()
            model, o, info = D.train_fold(make_model(), Xa, y[a], Xb, y[b], cfg, seed=s * 100 + f,
                                          log_prefix=f"{name} f{f} s{s} ")
            oof[b] += o / len(SEEDS)
            test_pred += D.predict(model, Xt) / (F.N_FOLDS * len(SEEDS))
            fold_info.append({"fold": f, "seed": s, "best_epoch": info["best_epoch"], "auc": float(info["best_auc"])})
            log(f"{name} fold {f} seed {s}: AUC={info['best_auc']:.5f} best_epoch={info['best_epoch']} "
                f"({time.time() - t0:.0f}s)")
        del model
        torch.cuda.empty_cache()
    results[out_name] = F.save_predictions(out_name, train_id, oof, test_id, test_pred, y, fold,
                                           extra={"source": "3.3", "experiment": name, "seeds": SEEDS,
                                                  "config": cfg.__dict__, "folds": fold_info})
    m = results[out_name]
    log(f"==> {out_name}: OOF AUC={m['oof_roc_auc']:.5f} | AP={m['oof_average_precision']:.5f} "
        f"| fold std={m['fold_roc_auc_std']:.5f}")

In [ ]:
# ---------- ghép seed: trung bình xác suất có trọng số theo số seed của mỗi phần
for out_name, parts in MERGE.items():
    names = [p for p, _ in parts]
    k = np.array([n for _, n in parts], dtype=float)
    O_p, T_p = F.load_predictions(names, train_id, test_id)
    oof, test_pred = O_p @ (k / k.sum()), T_p @ (k / k.sum())
    results[out_name] = F.save_predictions(out_name, train_id, oof, test_id, test_pred, y, fold,
                                           extra={"source": "3.3 merge", "parts": parts})
    m = results[out_name]
    log(f"==> {out_name} (merge {parts}): OOF AUC={m['oof_roc_auc']:.5f} | fold mean={m['fold_roc_auc_mean']:.5f}")

In [ ]:
# ---------- So với ML: AUC, Spearman với lgb_nl3, và blend ML + DL với trọng số DL chọn bằng CV theo fold
from scipy.stats import spearmanr

ML = ["lgb_nl3", "lgb_nl4", "lgb_nl8", "spline_lr", "xgb"]
O_ml, _ = F.load_predictions(ML, train_id, test_id)
R_ml = np.column_stack([F.rank01(O_ml[:, i]) for i in range(len(ML))])
w_ml, _ = F.tune_weights(R_ml, y)
w_ml = 0.7 * w_ml + 0.3 / len(ML)
ml_blend = F.rank01(R_ml @ w_ml)
ml_fold_auc = np.array([roc_auc_score(y[fold == f], ml_blend[fold == f]) for f in range(F.N_FOLDS)])

ALPHAS = np.round(np.arange(0, 0.51, 0.025), 3)

def cv_blend(dl_rank):
    """Chọn alpha trên 4 fold, chấm trên fold còn lại. Trả về (AUC trung bình theo fold, alpha từng fold)."""
    per_fold_alpha = np.array([[roc_auc_score(y[fold == f], (1 - al) * ml_blend[fold == f] + al * dl_rank[fold == f])
                                for f in range(F.N_FOLDS)] for al in ALPHAS])   # (n_alpha, n_fold)
    scores, chosen = [], []
    for f in range(F.N_FOLDS):
        others = [g for g in range(F.N_FOLDS) if g != f]
        best = int(np.argmax(per_fold_alpha[:, others].mean(1)))
        chosen.append(float(ALPHAS[best]))
        scores.append(per_fold_alpha[best, f])
    return float(np.mean(scores)), chosen

DL = []
for n in dict.fromkeys([*PREVIOUS, *results]):          # bỏ qua model chưa có output được gắn
    try:
        F.find_prediction("oof", n)
        DL.append(n)
    except FileNotFoundError:
        print("skip (không tìm thấy):", n)
O_dl, _ = F.load_predictions(DL, train_id, test_id)
rows = []
for i, n in enumerate(DL):
    d = O_dl[:, i]
    cv_auc, alphas = cv_blend(F.rank01(d))
    rows.append({"model": n, "oof_auc": roc_auc_score(y, d),
                 "spearman_vs_lgb_nl3": spearmanr(d, O_ml[:, 0]).correlation,
                 "ml_only_fold_auc": ml_fold_auc.mean(), "ml+dl_cv_fold_auc": cv_auc,
                 "gain": cv_auc - ml_fold_auc.mean(), "alpha_per_fold": alphas})
cmp = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 200)
print(cmp.round(5))
cmp.to_csv("3.ENSEMBLE/results/dl_vs_ml_3_3.csv")
log("done")

In [ ]:
# Gom dự đoán DL của các phiên bản 3.3 trước vào output này, để phiên bản mới nhất chứa đủ mọi model cho 3.4.
# Không chép mlp_base (đã có ở output 3.2) để tránh trùng tên khi 3.4 gắn cả hai.
import shutil
for n in DL:
    if n == "mlp_base" or (Path("3.ENSEMBLE/oof") / f"{n}.csv").exists():
        continue
    for kind in ("oof", "test"):
        shutil.copy(F.find_prediction(kind, n), Path("3.ENSEMBLE") / kind / f"{n}.csv")
    src = F.find_prediction("oof", n).parent.parent / "results" / f"{n}_metrics.json"
    if src.exists():
        shutil.copy(src, Path("3.ENSEMBLE/results") / src.name)
    print("carried over:", n)
print(sorted(p.stem for p in Path("3.ENSEMBLE/oof").glob("*.csv")))